# Refresh the OpenLineage ingestion token

The Spark OpenLineage HTTP transport authenticates to Eventhouse streaming
ingestion with a Kusto access token baked into the `lineage-env` Spark
Environment (`spark.openlineage.transport.auth.apiKey`). That token expires
(typically within an hour), so this notebook refreshes it:

1. Resolves the current workspace and finds the `lineage-env` Environment.
2. Gets a fresh Kusto token for the notebook's own identity.
3. Patches only the `apiKey` in the Environment's staged Spark properties.
4. Publishes the Environment so the new token takes effect.

**Schedule it in Fabric** (Run -> Schedule, e.g. hourly) so a long-lived demo
keeps ingesting without anyone's laptop being involved. The notebook runs under
its scheduling identity, which must have ingestor rights on the Eventhouse and
contributor on the workspace. Running it under your own user identity (the
default for a schedule you create) already satisfies this.

In [ ]:
# Parameters (overridable by a scheduled run). Defaults match deploy.ps1.
ENV_NAME    = "lineage-env"
FABRIC_BASE = "https://api.fabric.microsoft.com/v1"
# Optional explicit ids; leave blank to auto-discover from the running workspace.
WORKSPACE_ID   = ""
ENVIRONMENT_ID = ""

In [ ]:
import time
import requests


def _ctx():
    # notebookutils is the current Fabric global; mssparkutils is the legacy fallback.
    try:
        return notebookutils.runtime.context        # noqa: F821
    except NameError:
        return mssparkutils.runtime.context          # noqa: F821


def _token(audience):
    try:
        return notebookutils.credentials.getToken(audience)   # noqa: F821
    except NameError:
        return mssparkutils.credentials.getToken(audience)    # noqa: F821


def kusto_token():
    return _token("kusto")


def fabric_token():
    return _token("pbi")


if not WORKSPACE_ID:
    WORKSPACE_ID = _ctx().get("currentWorkspaceId")
print("workspace:", WORKSPACE_ID)

In [ ]:
# Find the lineage-env Environment id (unless one was passed in).
h = {"Authorization": "Bearer " + fabric_token()}
if not ENVIRONMENT_ID:
    r = requests.get(f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/environments", headers=h)
    r.raise_for_status()
    envs = r.json().get("value", [])
    match = [e for e in envs if e.get("displayName") == ENV_NAME]
    if not match:
        raise RuntimeError(f"Environment '{ENV_NAME}' not found in workspace {WORKSPACE_ID}.")
    ENVIRONMENT_ID = match[0]["id"]
print("environment:", ENVIRONMENT_ID)

In [ ]:
# Patch only the apiKey, preserving every other staged Spark property, then publish.
base = f"{FABRIC_BASE}/workspaces/{WORKSPACE_ID}/environments/{ENVIRONMENT_ID}"
h_json = {"Authorization": "Bearer " + fabric_token(), "Content-Type": "application/json"}

staging = requests.get(base + "/staging/sparkcompute", headers=h_json)
staging.raise_for_status()
current = staging.json() or {}
props = dict(current.get("sparkProperties") or {})
props["spark.openlineage.transport.auth.apiKey"] = kusto_token()

body = {"sparkProperties": props}
if current.get("instancePool"):
    body["instancePool"] = current["instancePool"]
if current.get("dynamicExecutorAllocation") is not None:
    body["dynamicExecutorAllocation"] = current["dynamicExecutorAllocation"]

r = requests.patch(base + "/staging/sparkcompute", headers=h_json, json=body)
print("patch sparkcompute HTTP", r.status_code, r.text[:300])
r.raise_for_status()

pub = requests.post(base + "/staging/publish", headers=h_json, json={})
print("publish HTTP", pub.status_code, pub.text[:300])
pub.raise_for_status()

In [ ]:
# Poll until the publish finishes.
final_state = None
for _ in range(120):
    state = requests.get(base, headers={"Authorization": "Bearer " + fabric_token()}).json()
    details = (state.get("properties") or {}).get("publishDetails") or {}
    final_state = details.get("state")
    print("publish state:", final_state)
    if final_state in ("Success", "Failed", "Cancelled"):
        break
    time.sleep(15)
if final_state != "Success":
    raise RuntimeError("Environment publish did not succeed: state=" + str(final_state))
print("OpenLineage token refreshed.")